In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))  # add repo root
from rsr import rsr
import torch
import json

from ndtools import fun_binary_graph as fbg # ndtools available at github.com/jieunbyun/network-datasets
from ndtools.graphs import build_graph
from pathlib import Path
import networkx as nx   


# Load data

In [2]:
DATASET = Path("data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs.json").read_text(encoding="utf-8"))

# build base graph
G_base: nx.Graph = build_graph(nodes, edges, probs_dict)

# all edges ON (example); add node/edge 0s as needed
states = {eid: 1 for eid in edges.keys()}

k_val, sys_st, _ = fbg.eval_global_conn_k(states, G_base)
print("k =", k_val, "System state =", sys_st)

k = 2 System state = 2


In [3]:
s_fun = lambda comps_st: fbg.eval_global_conn_k(comps_st, G_base)
row_names = list(edges.keys())
n_state = 2  # binary states: 0, 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

In [ ]:
RSRPATH = Path("rsr_res") 
sys_upper_st = 2 # either 1 or 2

refs_mat_upper = torch.load(RSRPATH / f"refs_up_{sys_upper_st}.pt", map_location="cpu")
refs_mat_upper = refs_mat_upper.to(device)
refs_mat_lower = torch.load(RSRPATH / f"refs_low_{sys_upper_st-1}.pt", map_location="cpu")
refs_mat_lower = refs_mat_lower.to(device)

FileNotFoundError: [Errno 2] No such file or directory: 'rsr_res\\rules_geq_2.pt'

# Calculate system probabilities

## Marginal probability

In [ ]:
pr_cond = rsr.get_comp_cond_sys_prob(
    refs_mat_upper,
    refs_mat_lower,
    probs,
    comps_st_cond = {},
    row_names = row_names,
    s_fun = s_fun,
    sys_upper_st = sys_upper_st
)
print(f"P(sys >= {sys_upper_st}) = {pr_cond['survival']:.3f}")
print(f"P(sys <= {sys_upper_st-1}) = {pr_cond['failure']:.3f}\n")

P(sys >= 2) = 0.168
P(sys <= 1) = 0.832



## Conditional probability given "one" component's survival

In [ ]:
for x in row_names[:-1]:
    print(f"Eval P(sys | {x}=1)")
    pr_cond = rsr.get_comp_cond_sys_prob(
        refs_mat_upper,
        refs_mat_lower,
        probs,
        comps_st_cond = {x: 1},
        row_names = row_names,
        s_fun = s_fun,
        sys_upper_st = sys_upper_st
    )
    print(f"P(sys >= {sys_upper_st} | {x}=1) = {pr_cond['survival']:.3f}")
    print(f"P(sys <= {sys_upper_st-1} | {x}=1) = {pr_cond['failure']:.3f}\n")

Eval P(sys | e01=1)
P(sys >= 2 | e01=1) = 0.168
P(sys <= 1 | e01=1) = 0.832

Eval P(sys | e02=1)
P(sys >= 2 | e02=1) = 0.210
P(sys <= 1 | e02=1) = 0.790

Eval P(sys | e03=1)
P(sys >= 2 | e03=1) = 0.209
P(sys <= 1 | e03=1) = 0.791

Eval P(sys | e04=1)
P(sys >= 2 | e04=1) = 0.209
P(sys <= 1 | e04=1) = 0.791

Eval P(sys | e05=1)
P(sys >= 2 | e05=1) = 0.210
P(sys <= 1 | e05=1) = 0.790

Eval P(sys | e06=1)
P(sys >= 2 | e06=1) = 0.168
P(sys <= 1 | e06=1) = 0.832

Eval P(sys | e07=1)
P(sys >= 2 | e07=1) = 0.209
P(sys <= 1 | e07=1) = 0.791

Eval P(sys | e08=1)
P(sys >= 2 | e08=1) = 0.210
P(sys <= 1 | e08=1) = 0.790

Eval P(sys | e09=1)
P(sys >= 2 | e09=1) = 0.210
P(sys <= 1 | e09=1) = 0.790

Eval P(sys | e10=1)
P(sys >= 2 | e10=1) = 0.168
P(sys <= 1 | e10=1) = 0.832



## Conditional probability given "two" component state

In [ ]:
for i, x in enumerate(row_names[:-1]):
    for j, y in enumerate(row_names[i+1:-1]):
        
        print(f"Eval P(sys | {x}=1, {y}=1)")
        pr_cond = rsr.get_comp_cond_sys_prob(
            refs_mat_upper,
            refs_mat_lower,
            probs,
            comps_st_cond = {x: 1, y: 1},
            row_names = row_names,
            s_fun = s_fun,
            sys_upper_st = sys_upper_st
        )
        print(f"P(sys >= {sys_upper_st} | {x}=1, {y}=1) = {pr_cond['survival']:.3f}")
        print(f"P(sys <= {sys_upper_st-1} | {x}=1, {y}=1) = {pr_cond['failure']:.3f}\n")
   

Eval P(sys | e01=1, e02=1)
P(sys >= 2 | e01=1, e02=1) = 0.209
P(sys <= 1 | e01=1, e02=1) = 0.791

Eval P(sys | e01=1, e03=1)
P(sys >= 2 | e01=1, e03=1) = 0.209
P(sys <= 1 | e01=1, e03=1) = 0.791

Eval P(sys | e01=1, e04=1)
P(sys >= 2 | e01=1, e04=1) = 0.210
P(sys <= 1 | e01=1, e04=1) = 0.790

Eval P(sys | e01=1, e05=1)
P(sys >= 2 | e01=1, e05=1) = 0.210
P(sys <= 1 | e01=1, e05=1) = 0.790

Eval P(sys | e01=1, e06=1)
P(sys >= 2 | e01=1, e06=1) = 0.168
P(sys <= 1 | e01=1, e06=1) = 0.832

Eval P(sys | e01=1, e07=1)
P(sys >= 2 | e01=1, e07=1) = 0.210
P(sys <= 1 | e01=1, e07=1) = 0.790

Eval P(sys | e01=1, e08=1)
P(sys >= 2 | e01=1, e08=1) = 0.209
P(sys <= 1 | e01=1, e08=1) = 0.791

Eval P(sys | e01=1, e09=1)
P(sys >= 2 | e01=1, e09=1) = 0.209
P(sys <= 1 | e01=1, e09=1) = 0.791

Eval P(sys | e01=1, e10=1)
P(sys >= 2 | e01=1, e10=1) = 0.168
P(sys <= 1 | e01=1, e10=1) = 0.832

Eval P(sys | e02=1, e03=1)
P(sys >= 2 | e02=1, e03=1) = 0.262
P(sys <= 1 | e02=1, e03=1) = 0.738

Eval P(sys | e02=1, 